# NDT Congestion Classifier — Training Notebook

**Predictive Network Digital Twin (DA/DS Module)**

This notebook walks through:
1. Exploratory Data Analysis (EDA) on the synthetic telemetry dataset
2. Feature engineering
3. XGBoost model training + hyperparameter discussion
4. Evaluation: accuracy, F1-score, confusion matrix, feature importance
5. Model export for the FastAPI service

---
> **Run order:** First execute `python data/generate_data.py` from `ml-engine/` to create the CSV.

In [ ]:
import os, sys
# Add ml-engine root to path so we can import train.py utilities
ML_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ML_ROOT not in sys.path:
    sys.path.insert(0, ML_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Imports OK')

## 1. Load Data

In [ ]:
DATA_PATH = os.path.join(ML_ROOT, 'data', 'network_telemetry.csv')

if not os.path.exists(DATA_PATH):
    import subprocess
    print('Generating data …')
    subprocess.run([sys.executable, os.path.join(ML_ROOT, 'data', 'generate_data.py')], check=True)

df = pd.read_csv(DATA_PATH)
if 'utilization_ratio' not in df.columns:
    df['utilization_ratio'] = df['throughput_mbps'] / df['capacity_mbps']

print(f'Dataset shape: {df.shape}')
df.head()

## 2. EDA

In [ ]:
print('Class distribution:')
print(df['congestion_status'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

CLASS_ORDER = ['Uncongested', 'Balanced', 'Moderately Congested', 'Highly Congested']
PALETTE = {'Uncongested': '#22d3ee', 'Balanced': '#84cc16',
           'Moderately Congested': '#f97316', 'Highly Congested': '#ef4444'}

# Throughput distribution by class
for label in CLASS_ORDER:
    subset = df[df['congestion_status'] == label]['throughput_mbps']
    axes[0].hist(subset, bins=40, alpha=0.6, label=label, color=PALETTE[label])
axes[0].set_title('Throughput by Class')
axes[0].set_xlabel('throughput_mbps')
axes[0].legend(fontsize=7)

# Delay distribution by class
for label in CLASS_ORDER:
    subset = df[df['congestion_status'] == label]['delay_ms']
    axes[1].hist(subset, bins=40, alpha=0.6, label=label, color=PALETTE[label])
axes[1].set_title('Delay by Class')
axes[1].set_xlabel('delay_ms')
axes[1].legend(fontsize=7)

# Utilisation ratio distribution
for label in CLASS_ORDER:
    subset = df[df['congestion_status'] == label]['utilization_ratio']
    axes[2].hist(subset, bins=40, alpha=0.6, label=label, color=PALETTE[label])
axes[2].set_title('Utilisation Ratio by Class')
axes[2].set_xlabel('utilization_ratio')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric = df[['throughput_mbps', 'capacity_mbps', 'delay_ms', 'utilization_ratio']]
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Prepare Features & Labels

In [ ]:
FEATURES = ['throughput_mbps', 'capacity_mbps', 'delay_ms', 'utilization_ratio']
TARGET   = 'congestion_status'

le = LabelEncoder()
le.fit(CLASS_ORDER)
y = le.transform(df[TARGET])
X = df[FEATURES].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 4. Train XGBoost Pipeline

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        objective='multi:softprob',
        num_class=4,
        random_state=42,
        n_jobs=-1,
    ))
])

pipe.fit(X_train, y_train)
print('Training complete!')

## 5. Evaluate

In [ ]:
y_pred      = pipe.predict(X_test)
y_pred_lbl  = le.inverse_transform(y_pred)
y_test_lbl  = le.inverse_transform(y_test)

acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}  ({acc*100:.2f} %)')
print()
print(classification_report(y_test_lbl, y_pred_lbl, target_names=CLASS_ORDER))

In [ ]:
# 5-fold CV
cv = cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv.mean():.4f} ± {cv.std():.4f}')

In [ ]:
# Confusion matrix
cm   = confusion_matrix(y_test_lbl, y_pred_lbl, labels=CLASS_ORDER)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — NDT Congestion Classifier', pad=12)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pipe.named_steps['clf'].feature_importances_
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(FEATURES, importances, color=['#22d3ee','#84cc16','#f97316','#6366f1'])
ax.set_title('XGBoost Feature Importances')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, importances):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
MODEL_DIR  = os.path.join(ML_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(pipe, os.path.join(MODEL_DIR, 'congestion_model.joblib'))
joblib.dump(le,   os.path.join(MODEL_DIR, 'label_encoder.joblib'))

print('Model saved to models/congestion_model.joblib')
print('Encoder saved to models/label_encoder.joblib')

## 7. Quick Inference Test

In [ ]:
samples = np.array([
    [900.0,  1000, 45.2, 0.90],   # expected: Highly Congested
    [400.0,  1000, 10.1, 0.40],   # expected: Balanced
    [50.0,    500,  4.5, 0.10],   # expected: Uncongested
    [700.0,  1000, 22.5, 0.70],   # expected: Moderately Congested
])

preds = le.inverse_transform(pipe.predict(samples))
probs = pipe.predict_proba(samples).max(axis=1)

print(f'{"throughput":>12} {"capacity":>10} {"delay":>8} {"util":>6}  →  {"prediction":<25}  confidence')
print('-' * 80)
for feat, lbl, prob in zip(samples, preds, probs):
    t, c, d, u = feat
    print(f'{t:12.0f} {c:10.0f} {d:8.1f} {u:6.0%}  →  {lbl:<25}  {prob:.3f}')